In [ ]:
from pathlib import Path
import pandas as pd
from typing import Literal

In [ ]:
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
CHUNK_SIZE = 500_000
DATASET = {
    "payments": "General_Payments_2024.csv",
    "prescribers": "Med_Prescribers_2024.csv",
}
CLEANED_SET ={
    "company": "cleaned_companies.csv",
    "doctor": "cleaned_doctors.csv",
    "prescription": "cleaned_prescriptions.csv",
    "drug": "cleaned_drugs.csv"
}

In [ ]:
# Load file into DataFrames using a chunk size to limit the number of rows read
def load_in_chunks(file_path, chunk_size, file_type: Literal["json", "csv", "xlsx", "xls"]="csv") -> pd.DataFrame:
    try:
        chunks = []
        chunk_no = 0
        if file_type == "json":
            for chunk in pd.read_json(file_path, lines=True, chunksize=chunk_size):
                # print(f"Chunk {chunk_no + 1} loaded. Shape: {chunk.shape}")
                chunks.append(chunk)
                chunk_no += 1
        elif file_type == "csv":
            for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
                # print(f"Chunk {chunk_no + 1} loaded. Shape: {chunk.shape}")
                chunks.append(chunk)
                chunk_no += 1
        elif file_type in ["xlsx", "xls"]:
            for chunk in pd.read_excel(file_path, chunksize=chunk_size):
                # print(f"Chunk {chunk_no + 1} loaded. Shape: {chunk.shape}")
                chunks.append(chunk)
                chunk_no += 1
        return pd.concat(chunks, ignore_index=True)
    except Exception as e:
        print(f"Error loading file {file_path}: {e}")
        return pd.DataFrame()

def get_file_extension(file_path: str) -> str:
    try:
        return Path(file_path).suffix.lower().lstrip(".")
    except Exception as e:
        print(f"Error getting file extension for {file_path}: {e}")
        return ""


In [4]:
payments_path = DATA_DIR / DATASET["payments"]
prescribers_path = DATA_DIR / DATASET["prescribers"]

payments_df = load_in_chunks(payments_path, CHUNK_SIZE, file_type=get_file_extension(payments_path))
prescribers_df = load_in_chunks(prescribers_path, CHUNK_SIZE, file_type=get_file_extension(prescribers_path))

In [5]:
# Print the shape and first few rows of the payments dataframe
print(payments_df.shape)
payments_df.head(3)

(15498687, 91)


,Change_Type,Covered_Recipient_Type,Teaching_Hospital_CCN,Teaching_Hospital_ID,Teaching_Hospital_Name,Covered_Recipient_Profile_ID,Covered_Recipient_NPI,Covered_Recipient_First_Name,Covered_Recipient_Middle_Name,Covered_Recipient_Last_Name,...,Associated_Drug_or_Biological_NDC_4,Associated_Device_or_Medical_Supply_PDI_4,Covered_or_Noncovered_Indicator_5,Indicate_Drug_or_Biological_or_Device_or_Medical_Supply_5,Product_Category_or_Therapeutic_Area_5,Name_of_Drug_or_Biological_or_Device_or_Medical_Supply_5,Associated_Drug_or_Biological_NDC_5,Associated_Device_or_Medical_Supply_PDI_5,Program_Year,Payment_Publication_Date
0,UNCHANGED,Covered Recipient Non-Physician Practitioner,NaN,NaN,NaN,10593686.0,1.790251e+09,AMY,NaN,PONZIO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,06/30/2026
1,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,232750.0,1.134116e+09,CHRISTOPHER,NaN,STALLTER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,06/30/2026
2,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,9569889.0,1.417454e+09,ANDREA,NaN,TAI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,06/30/2026


In [6]:
# Print the shape and first few rows of the prescribers dataframe
print(prescribers_df.shape)
prescribers_df.head(3)

(28023892, 22)


,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src,Brnd_Name,Gnrc_Name,...,Tot_Day_Suply,Tot_Drug_Cst,Tot_Benes,GE65_Sprsn_Flag,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes
0,1003000126,Enkeshafi,Ardalan,Bethesda,MD,24,Internal Medicine,Claim-Specialty,Abrysvo,"Rsv Vacc, Pref A And Pref B/Pf",...,20,5994.01,20.0,NaN,20.0,20.0,5994.01,20.0,NaN,20.0
1,1003000126,Enkeshafi,Ardalan,Bethesda,MD,24,Internal Medicine,Claim-Specialty,Boostrix Tdap,"Diphth,pertuss(Acell),tet Vac",...,12,648.26,12.0,NaN,12.0,12.0,648.26,12.0,NaN,12.0
2,1003000142,Khalil,Rashid,Toledo,OH,39,Anesthesiology,Claim-Specialty,Acetaminophen-Codeine,Acetaminophen With Codeine,...,628,362.48,11.0,#,NaN,NaN,NaN,NaN,*,NaN


In [7]:
# Check for duplicated rows in the prescribers dataframe based on NPI, Brand Name, and Generic Name
prescribers_df.duplicated(
    ["Prscrbr_NPI", "Brnd_Name", "Gnrc_Name"]
).sum()

0

In [8]:
key_columns = ["Prscrbr_NPI", "Brnd_Name", "Gnrc_Name"]

# Missing values in each candidate key column
print(prescribers_df[key_columns].isna().sum())

# Total rows with at least one missing key value
print(
    "Rows with a missing key value:",
    prescribers_df[key_columns].isna().any(axis=1).sum()
)

Prscrbr_NPI    0
Brnd_Name      0
Gnrc_Name      0
dtype: int64
Rows with a missing key value: 0


In [9]:
print(payments_df.info())
print(prescribers_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 15498687 entries, 0 to 15498686
Data columns (total 91 columns):
 #   Column                                                             Dtype  
---  ------                                                             -----  
 0   Change_Type                                                        str    
 1   Covered_Recipient_Type                                             str    
 2   Teaching_Hospital_CCN                                              float64
 3   Teaching_Hospital_ID                                               float64
 4   Teaching_Hospital_Name                                             str    
 5   Covered_Recipient_Profile_ID                                       float64
 6   Covered_Recipient_NPI                                              float64
 7   Covered_Recipient_First_Name                                       str    
 8   Covered_Recipient_Middle_Name                                      str    
 9   Covered_Rec

In [10]:
# Drop rows with missing "Covered_Recipient_NPI" values and ensure the column is of integer type for consistency with prescribers_df.
print("Covered_Recipient_NPI Data Type:" ,payments_df["Covered_Recipient_NPI"].dtype)
print("Prscrbr_NPI Data Type:" ,prescribers_df["Prscrbr_NPI"].dtype)
print("Count of missing Covered_Recipient_NPI:", payments_df["Covered_Recipient_NPI"].isna().sum())

payments_df.dropna(subset=["Covered_Recipient_NPI"], inplace=True)
payments_df["Covered_Recipient_NPI"] = payments_df["Covered_Recipient_NPI"].astype("int64")


Covered_Recipient_NPI Data Type: float64
Prscrbr_NPI Data Type: int64
Count of missing Covered_Recipient_NPI: 51278


In [11]:
# Collect Company Information
company_df = (
    payments_df[[
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country",
    ]]
    .drop_duplicates(subset=["Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID"])
    .reset_index(drop=True)
).rename(columns={
    "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID": "company_id",
    "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name": "company_name",
    "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State": "state",
    "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Country": "country",
})


# Ensure that each company has a unique Company_ID in the final company dimension.
assert company_df["company_id"].is_unique
company_df.head()

,company_id,company_name,state,country
0,100000010419,Phadia US Inc.,MI,United States
1,100000151633,"Ancora Heart, Inc.",CA,United States
2,100000005674,Boston Scientific Corporation,MA,United States
3,100000961840,"Channel Medsystems, Inc.",CA,United States
4,100000010788,"One Lambda, Inc.",CA,United States


##### DOCTOR RESOLUTION

In [12]:
# Check for NPIs prescribing to multiple states. If none exists, then the data is consistent with single-state prescribing.
# This is to validate that the state in Medicare Data corresponds to the NPIs actual state of practice
states_per_npi = prescribers_df.groupby("Prscrbr_NPI")["Prscrbr_State_Abrvtn"].nunique()
multi_state = (states_per_npi > 1).sum()

print("NPIs prescribing to multiple states:", multi_state)
print("Check passed" if multi_state == 0 else "Check failed")

NPIs prescribing to multiple states: 0
Check passed


In [13]:
# Collect Prescriber/Doctor information and resolve specialty from payment data. Inner join used to avoid unmatched doctors. i.e Only consider doctors with payment records
pres = prescribers_df[[
    "Prscrbr_NPI",
    "Prscrbr_Last_Org_Name",
    "Prscrbr_First_Name",
    "Prscrbr_State_Abrvtn"
]].drop_duplicates(subset=["Prscrbr_NPI"])

pay = payments_df[[
    "Covered_Recipient_NPI",
    "Covered_Recipient_Specialty_1",
    "Covered_Recipient_Primary_Type_1",
    "Recipient_Primary_Business_Street_Address_Line1",
    "Recipient_Primary_Business_Street_Address_Line2"
]].drop_duplicates(subset=["Covered_Recipient_NPI"])


doctor_df = pres.merge(
    pay,
    left_on="Prscrbr_NPI",
    right_on="Covered_Recipient_NPI",
    how="inner"
).reset_index(drop=True)


doctor_df = doctor_df[[
    "Prscrbr_NPI",
    "Prscrbr_Last_Org_Name",
    "Prscrbr_First_Name",
    "Prscrbr_State_Abrvtn",
    "Covered_Recipient_Specialty_1",
    "Covered_Recipient_Primary_Type_1",
    "Recipient_Primary_Business_Street_Address_Line1",
    "Recipient_Primary_Business_Street_Address_Line2"
]].rename(columns={
    "Prscrbr_NPI": "npi",
    "Prscrbr_Last_Org_Name": "last_name",
    "Prscrbr_First_Name": "first_name",
    "Prscrbr_Type_Src": "provider_type",
    "Prscrbr_State_Abrvtn": "state",
    "Covered_Recipient_Specialty_1": "specialty",
    "Covered_Recipient_Primary_Type_1": "primary_type",
    "Recipient_Primary_Business_Street_Address_Line1": "address_line1",
    "Recipient_Primary_Business_Street_Address_Line2": "address_line2"
})

# Ensure that each doctor has a unique NPI in the final doctor dimension.
assert doctor_df["npi"].is_unique
assert doctor_df["state"].notnull().all()

print("Doctors Row Count:", doctor_df.shape[0])
doctor_df.head()



Doctors Row Count: 642414


,npi,last_name,first_name,state,specialty,primary_type,address_line1,address_line2
0,1003000142,Khalil,Rashid,OH,Allopathic & Osteopathic Physicians|Anesthesio...,Medical Doctor,4126 N HOLLAND SYLVANIA RD,SUITE 220
1,1003000167,Escobar,Julio,NV,Dental Providers|Dentist,Doctor of Dentistry,5 PINE CONE RD,NaN
2,1003000423,Velotta,Jennifer,OH,Allopathic & Osteopathic Physicians|Obstetrics...,Medical Doctor,9500 Mentor Ave,Ste 220
3,1003000480,Rothchild,Kevin,CO,Allopathic & Osteopathic Physicians|Surgery|Tr...,Medical Doctor,12605 E 16TH AVE,NaN
4,1003000530,Semonche,Amanda,PA,Allopathic & Osteopathic Physicians|Internal M...,Doctor of Osteopathy,1021 PARK AVE,SUITE 203


##### DRUG RESOLUTION
Because Brand Name maps to more than one Generic Name, it will be impossible to use a brand name as the primary key. 

Will use a surrogate key "drug_id" as the primary key and keep both Brnd_Name and Gnrc_Name.

In [14]:
# Collect unique drug information from prescribers_df.
drug_dim = prescribers_df[["Brnd_Name", "Gnrc_Name", "Tot_Clms"]].copy()

# Normalize text so case/whitespace variants don't create false duplicates.
for col in ["Brnd_Name", "Gnrc_Name"]:
    drug_dim[col] = drug_dim[col].str.strip()

print("Missing:", drug_dim[["Brnd_Name", "Gnrc_Name"]].isna().sum().to_dict())
drug_dim = drug_dim.dropna(subset=["Brnd_Name", "Gnrc_Name"])

# Brand names that map to more than one generic name violate the PRIMARY KEY.
generics_per_brand = drug_dim.groupby("Brnd_Name")["Gnrc_Name"].nunique()
conflicts = generics_per_brand[generics_per_brand > 1].index
print("Brands:", generics_per_brand.shape[0], "| Brands with >1 generic:", len(conflicts))
drug_dim[drug_dim["Brnd_Name"].isin(conflicts)].groupby(["Brnd_Name", "Gnrc_Name"])["Tot_Clms"].sum().head(20)


Missing: {'Brnd_Name': 0, 'Gnrc_Name': 0}
Brands: 3055 | Brands with >1 generic: 74


Brnd_Name                       Gnrc_Name                     
Advocate Syringes               Syring-Needl,disp,insul,0.3 Ml      1576
                                Syringe And Needle,insulin,1ml      4827
                                Syringe-Needle,insulin,0.5 Ml       5264
Akynzeo                         Fosnetupitant/Palonosetron          4152
                                Netupitant/Palonosetron Hcl          200
Atropine Sulfate                Atropine Sulfate                  122812
                                Atropine Sulfate/Pf                   53
Bd Insulin Syringe Ult-Fine Ii  Syring-Needl,disp,insul,0.3 Ml        62
                                Syringe-Needle,insulin,0.5 Ml         73
Betamethasone Diprop Augmented  Betamethasone Dipropionate          1498
                                Betamethasone/Propylene Glyc      226005
Bupivacaine Hcl                 Bupivacaine Hcl                       26
                                Bupivacaine Hcl/Pf           

In [15]:
# Using a surrogate key drug_id for the drug dimension. since brand_name is not unique and cannot be used as a primary key
drug_df = (
    drug_dim[["Brnd_Name", "Gnrc_Name"]]
    .drop_duplicates()
    .rename(columns={"Brnd_Name": "brand_name", "Gnrc_Name": "generic_name"})
    .reset_index(drop=True)
)
drug_df.insert(0, "drug_id", range(1, len(drug_df) + 1))

assert drug_df["drug_id"].is_unique
assert drug_df["brand_name"].notna().all()
assert drug_df["generic_name"].notna().all()

print("Drug Row Count:", drug_df.shape[0])
drug_df.head()

Drug Row Count: 3159


,drug_id,brand_name,generic_name
0,1,Abrysvo,"Rsv Vacc, Pref A And Pref B/Pf"
1,2,Boostrix Tdap,"Diphth,pertuss(Acell),tet Vac"
2,3,Acetaminophen-Codeine,Acetaminophen With Codeine
3,4,Amitriptyline Hcl,Amitriptyline Hcl
4,5,Baclofen,Baclofen


##### PRESCRIPTION RESOLUTION

In [16]:
prescription_df = (
    prescribers_df[["Prscrbr_NPI", "Tot_Clms", "Tot_Day_Suply", "Tot_Drug_Cst", "Tot_Benes", "Brnd_Name", "Gnrc_Name"]]
    .merge(drug_df, how="inner", left_on=["Brnd_Name", "Gnrc_Name"], right_on=["brand_name", "generic_name"])
    .reset_index(drop=True)[["Prscrbr_NPI", "drug_id", "Tot_Clms", "Tot_Day_Suply", "Tot_Drug_Cst", "Tot_Benes"]]
).rename(columns={"Prscrbr_NPI": "npi", "Tot_Clms": "total_claims", "Tot_Day_Suply": "total_day_supply", "Tot_Drug_Cst": "total_drug_cost", "Tot_Benes": "total_beneficiaries"})


# check that prescription has unique npi, drug_id combination
assert prescription_df.groupby(["npi", "drug_id"]).ngroups == prescription_df.shape[0]

print("Prescriptions Row Count: ", prescription_df.shape[0])
prescription_df.head()


Prescriptions Row Count:  28023892


,npi,drug_id,total_claims,total_day_supply,total_drug_cost,total_beneficiaries
0,1003000126,1,20,20,5994.01,20.0
1,1003000126,2,12,12,648.26,12.0
2,1003000142,3,24,628,362.48,11.0
3,1003000142,4,47,1410,271.17,16.0
4,1003000142,5,47,1395,333.35,26.0


In [17]:
def npi_overlap(prescribers_df: pd.DataFrame, payments_df: pd.DataFrame, states=None) -> dict:
    """Distinct NPI overlap between prescribers_df and payments_df, optionally limited to prescriber states."""
    if isinstance(states, str):
        states = [states]

    pres = prescribers_df
    if states:
        pres = pres[pres["Prscrbr_State_Abrvtn"].isin(states)]

    pres_npis = set(pres["Prscrbr_NPI"].unique())
    pay_npis = set(payments_df["Covered_Recipient_NPI"].unique())
    overlap = pres_npis & pay_npis

    result = {
        "States": states or "ALL",
        "distinct_prescriber_npis": len(pres_npis),
        "distinct_overlap_npis": len(overlap),
        "overlap_pct": round(len(overlap) / len(pres_npis) * 100, 2) if pres_npis else 0.0,
    }
    return result

In [18]:
print(npi_overlap(prescribers_df, payments_df, ["PA", "NJ", "DE"]))
print(npi_overlap(prescribers_df, payments_df, "NM"))
print(npi_overlap(prescribers_df, payments_df))

{'States': ['PA', 'NJ', 'DE'], 'distinct_prescriber_npis': 85867, 'distinct_overlap_npis': 50986, 'overlap_pct': 59.38}
{'States': ['NM'], 'distinct_prescriber_npis': 6781, 'distinct_overlap_npis': 3240, 'overlap_pct': 47.78}
{'States': 'ALL', 'distinct_prescriber_npis': 1139455, 'distinct_overlap_npis': 642414, 'overlap_pct': 56.38}


##### CSV OUTPUT

In [19]:
states_of_interest = ["NM"]

# Filter doctors and prescriptions for states of interest
doctor = doctor_df[doctor_df["state"].isin(states_of_interest)]
prescription = prescription_df.merge(
    doctor_df[doctor_df["state"].isin(states_of_interest)][["npi", "state"]],
    how="inner",
    on="npi",
)

print("Presciption by state of interest: ", prescription.shape[0])

# drug_df.to_csv(OUTPUT_DIR / CLEANED_SET["drug"], index=False) # Drug


Presciption by state of interest:  91302


In [20]:
# Analysis of prescribers and prescriptions for states of interest. Limiting to NPIs with payments reduces the dataset to relevant doctors
nm_rows = prescribers_df[
    prescribers_df["Prscrbr_State_Abrvtn"].isin(states_of_interest)
]
doctor_npis = doctor_df["npi"].dropna().unique()
nm_matches_doctor = nm_rows["Prscrbr_NPI"].isin(doctor_npis)

print("States of Interest prescriber rows:", len(nm_rows))
print("States of Interest distinct NPIs:", nm_rows["Prscrbr_NPI"].nunique())
print("States of Interest rows whose NPI is in doctor_df (Doctors with Payments):", nm_matches_doctor.sum())
print("Rows in prescription:", len(prescription))
print(
    "Distinct States of Interest NPIs in doctor_df:",
    nm_rows.loc[nm_matches_doctor, "Prscrbr_NPI"].nunique(),
)
print("Distinct States of Interest NPIs in prescription:", prescription["npi"].nunique())

States of Interest prescriber rows: 151371
States of Interest distinct NPIs: 6781
States of Interest rows whose NPI is in doctor_df (Doctors with Payments): 91302
Rows in prescription: 91302
Distinct States of Interest NPIs in doctor_df: 3240
Distinct States of Interest NPIs in prescription: 3240


In [21]:
# Export to csv if ok with the chosen states of interest
company_df.to_csv(OUTPUT_DIR / CLEANED_SET["company"], index=False)
doctor.to_csv(OUTPUT_DIR / CLEANED_SET["doctor"], index=False)
drug_df.to_csv(OUTPUT_DIR / CLEANED_SET["drug"], index=False)
prescription.to_csv(OUTPUT_DIR / CLEANED_SET["prescription"], index=False)